# LeetCode 438 — Find All Anagrams in a String bằng Python

Link đề: https://leetcode.com/problems/find-all-anagrams-in-a-string/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def findAnagrams(self, s: str, p: str) -> List[int]:
        ...
```

Ghi chú: đề và các ví dụ được viết lại theo hiểu biết về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (so với cách sắp xếp từng cửa sổ), nhưng nên đối chiếu lại đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **mọi vị trí bắt đầu** của một chuỗi con (liên tiếp) của `s` mà là **anagram** (hoán vị các chữ cái) của `p`.
2. Nắm các công cụ Python: lát cắt `s[i:j]`, `sorted`, `collections.Counter`, mảng đếm 26 phần tử với `ord(c) - ord('a')`, so sánh list/`dict`, và `random` có seed để tạo hash.
3. Đi từ **sắp xếp từng cửa sổ `O(m · n log n)`** → **cửa sổ trượt với hai bộ đếm** → **mảng 26 + đếm số chữ cái đang lệch (`O(m)`)** → **cửa sổ co giãn kiểu "còn thiếu"** → **hash cộng ngẫu nhiên rồi kiểm chứng lại**.
4. Hiểu mấu chốt: hai chuỗi là anagram ⇔ **cùng bảng đếm chữ cái**; cửa sổ có độ dài **cố định `len(p)`** trượt sang phải một ô chỉ **thêm 1 chữ, bớt 1 chữ**, nên bảng đếm cập nhật trong `O(1)`.
5. Tránh các lỗi kinh điển: không bớt chữ bên trái (cửa sổ phình ra), tính sai vị trí bắt đầu `i - n + 1`, so sánh `set` thay vì bảng đếm, và quên rằng `dict` giữ khoá có đếm `0` làm phép so sánh sai.

## 2. Đề bài & ví dụ

Cho hai chuỗi `s` và `p`. Hãy trả về mảng gồm **tất cả các chỉ số bắt đầu** của các chuỗi con trong `s` là **anagram** của `p`. Có thể trả về theo **thứ tự bất kỳ**.

**Anagram** của một chuỗi là chuỗi thu được khi **hoán đổi thứ tự** các chữ cái của nó (dùng đủ mọi chữ cái, đúng số lần). Ví dụ `"abc"`, `"bca"`, `"cab"` là anagram của nhau.

**Ví dụ 1**

- Input: `s = "cbaebabacd"`, `p = "abc"`
- Output: `[0, 6]`
- Giải thích: chuỗi con bắt đầu ở `0` là `"cba"` (anagram của `"abc"`); bắt đầu ở `6` là `"bac"` (anagram của `"abc"`).

**Ví dụ 2**

- Input: `s = "abab"`, `p = "ab"`
- Output: `[0, 1, 2]`
- Giải thích: `"ab"` (vị trí `0`), `"ba"` (vị trí `1`), `"ab"` (vị trí `2`) đều là anagram của `"ab"`.

**Ràng buộc:**

- `1 <= s.length, p.length <= 3 * 10^4`
- `s` và `p` chỉ gồm chữ cái tiếng Anh viết thường.

Chú ý: các chuỗi con được tìm **chồng lấn** được (ở ví dụ 2 các cửa sổ `[0,1]`, `[1,2]`, `[2,3]` dùng chung chữ cái).

In [1]:
from typing import List
from collections import Counter
import random
import string

examples = [
    ("cbaebabacd", "abc", [0, 6]),
    ("abab", "ab", [0, 1, 2]),
]
for s, p, expected in examples:
    print(repr(s), repr(p), "-> kỳ vọng", expected)

'cbaebabacd' 'abc' -> kỳ vọng [0, 6]
'abab' 'ab' -> kỳ vọng [0, 1, 2]


## 3. Kiến thức nền

### 3.1. Lát cắt chuỗi `s[i:j]`

`s[i:j]` lấy các ký tự có chỉ số từ `i` đến `j - 1` (**không** gồm `j`). Độ dài là `j - i`. Cửa sổ độ dài `n` bắt đầu tại `i` là `s[i:i + n]`.

In [2]:
s = "cbaebabacd"
print(s[0:3])          # cba
print(s[6:9])          # bac
print(s[2:2 + 3])      # aeb
print(s[8:11])         # cd (cắt vượt cuối thì chỉ lấy đến hết chuỗi, không báo lỗi)

cba
bac
aeb
cd


### 3.2. Anagram ⇔ cùng bảng đếm chữ cái (hoặc cùng chuỗi sau khi sắp)

Hai chuỗi là anagram khi mỗi chữ cái xuất hiện **số lần bằng nhau** trong cả hai. Hai cách kiểm tra:

- `sorted(a) == sorted(b)` (sắp rồi so);
- `Counter(a) == Counter(b)` (đếm rồi so).

In [3]:
print(sorted("cba"), sorted("abc"), sorted("cba") == sorted("abc"))       # True
print(Counter("cba") == Counter("abc"))                                     # True
print(sorted("aab") == sorted("abb"))                                       # False: số lần khác nhau
print(set("aab") == set("abb"))                                             # True!! set bỏ qua số lần -> không dùng được

['a', 'b', 'c'] ['a', 'b', 'c'] True
True
False
True


### 3.3. `Counter` — đếm số lần xuất hiện

`Counter("abca")` cho `{'a': 2, 'b': 1, 'c': 1}`. Tra một chữ chưa có trả `0`. `cnt[x] += 1` và `cnt[x] -= 1` thay đổi số đếm. Chú ý: giảm về `0` thì khoá vẫn nằm trong `Counter`/`dict` (số đếm `0`) — điều này liên quan tới phép so sánh sẽ nói ở mục lỗi thường gặp.

In [4]:
cnt = Counter("abca")
print(cnt, cnt["z"])             # Counter({'a': 2, 'b': 1, 'c': 1}) 0
cnt["a"] -= 2
print(cnt, "a" in cnt)           # khoá 'a' vẫn còn, với đếm 0

Counter({'a': 2, 'b': 1, 'c': 1}) 0
Counter({'b': 1, 'c': 1, 'a': 0}) True


### 3.4. Mảng đếm 26 phần tử: `ord(c) - ord('a')`

Vì chỉ có chữ thường, ta có thể dùng list 26 số: chữ `'a'` ứng với chỉ số `0`, `'b'` với `1`, ..., `'z'` với `25`. `ord('a') = 97`, nên chỉ số của `c` là `ord(c) - 97`. Hai list so sánh bằng `==` (so từng phần tử) trong `O(26)`.

In [5]:
def count26(text):
    cnt = [0] * 26
    for ch in text:
        cnt[ord(ch) - ord("a")] += 1
    return cnt


print(count26("abca")[:4])                       # [2, 1, 1, 0]
print(count26("abc") == count26("cab"))          # True
print(count26("abc") == count26("abd"))          # False

[2, 1, 1, 0]
True
False


### 3.5. Cửa sổ trượt (sliding window) độ dài cố định

Thay vì tính lại từ đầu cho mỗi vị trí, ta để cửa sổ `[i, i + n)` trượt sang phải một ô: **thêm** chữ `s[i + n]` ở bên phải, **bớt** chữ `s[i]` ở bên trái. Hai thao tác `O(1)`.

In [6]:
s, n = "cbaebabacd", 3
for i in range(len(s) - n + 1):
    print(i, s[i:i + n])

0 cba
1 bae
2 aeb
3 eba
4 bab
5 aba
6 bac
7 acd


### 3.6. Hash cộng và `random.Random(seed)`

Gán mỗi chữ cái một số ngẫu nhiên lớn cố định `w[c]`. "Hash" của một chuỗi là **tổng** `w` của các chữ. Hoán vị không đổi tổng (phép cộng giao hoán). Hai chuỗi khác bảng đếm gần như chắc chắn khác tổng (nhưng **không chắc chắn tuyệt đối**), nên khi tổng trùng ta vẫn kiểm chứng lại bằng bảng đếm. `random.Random(seed)` tạo bộ sinh riêng với seed cố định để kết quả lặp lại được.

In [7]:
rnd = random.Random(438)
w = {c: rnd.getrandbits(60) for c in string.ascii_lowercase}
h = lambda text: sum(w[c] for c in text)
print(h("abc") == h("cab"), h("abc") == h("abd"))     # True False

True False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `s = "cbaebabacd"`, `p = "abc"` (`n = 3`), bảng đếm của `p` là `{a:1, b:1, c:1}`. Ta xét từng cửa sổ độ dài `3`:

| Vị trí `i` | Cửa sổ | Bảng đếm | Bằng `p`? |
|---|---|---|---|
| 0 | `cba` | a1 b1 c1 | **có** |
| 1 | `bae` | a1 b1 e1 | không |
| 2 | `aeb` | a1 b1 e1 | không |
| 3 | `eba` | a1 b1 e1 | không |
| 4 | `bab` | a1 b2 | không |
| 5 | `aba` | a2 b1 | không |
| 6 | `bac` | a1 b1 c1 | **có** |
| 7 | `acd` | a1 c1 d1 | không |

Đáp án `[0, 6]`. Có `m - n + 1 = 10 - 3 + 1 = 8` cửa sổ. Mỗi cửa sổ khác cửa sổ trước đúng **một chữ vào, một chữ ra**: từ `cba` sang `bae` là bỏ `c`, thêm `e`.

In [8]:
s, p = "cbaebabacd", "abc"
n = len(p)
for i in range(len(s) - n + 1):
    w_ = s[i:i + n]
    print(i, w_, dict(sorted(Counter(w_).items())), "<- khớp" if Counter(w_) == Counter(p) else "")

0 cba {'a': 1, 'b': 1, 'c': 1} <- khớp
1 bae {'a': 1, 'b': 1, 'e': 1} 
2 aeb {'a': 1, 'b': 1, 'e': 1} 
3 eba {'a': 1, 'b': 1, 'e': 1} 
4 bab {'a': 1, 'b': 2} 
5 aba {'a': 2, 'b': 1} 
6 bac {'a': 1, 'b': 1, 'c': 1} <- khớp
7 acd {'a': 1, 'c': 1, 'd': 1} 


## 5. Cách 1 — Brute Force / baseline: sắp xếp từng cửa sổ

**Ý tưởng.** Sắp `p` một lần (`target = sorted(p)`). Với mỗi vị trí `i` từ `0` đến `m - n`, lấy `s[i:i+n]`, sắp xếp và so với `target`.

In [9]:
def anagrams_sort(s, p):
    m, n = len(s), len(p)
    target = sorted(p)
    ans = []
    for i in range(m - n + 1):
        if sorted(s[i:i + n]) == target:
            ans.append(i)
    return ans


for s, p, expected in examples:
    print(anagrams_sort(s, p), "(kỳ vọng", expected, ")")

[0, 6] (kỳ vọng [0, 6] )
[0, 1, 2] (kỳ vọng [0, 1, 2] )


**Độ phức tạp (Cách 1).** Có `m - n + 1` cửa sổ, mỗi cửa sổ cắt (`O(n)`) và sắp (`O(n log n)`) ⇒ **Time `O((m - n + 1) · n log n)`**, tệ nhất cỡ `O(m · n log n)` khi `n ~ m/2` (hàng tỉ thao tác với `m = 3·10^4`). **Space `O(n)`**.

**Nút thắt:** mỗi cửa sổ **tính lại từ đầu** dù nó chỉ khác cửa sổ trước đúng 2 chữ cái. Ta nên **cập nhật** thay vì tính lại.

## 6. Ý tưởng thuật toán / bước đột phá

**Giữ bảng đếm của cửa sổ và cập nhật `O(1)` mỗi bước.**

1. `need` = bảng đếm của `p` (không đổi).
2. `win` = bảng đếm của cửa sổ hiện tại (độ dài `n`).
3. Trượt sang phải: `win[s[i + n]] += 1` (chữ mới vào) và `win[s[i]] -= 1` (chữ cũ ra).
4. Nếu `win == need` thì `i + 1` (vị trí cửa sổ mới) là một đáp án.

So sánh hai bảng 26 ô tốn `O(26) = O(1)`. Tổng `O(26 · m)`. Còn có thể nhanh hơn nữa: chỉ cần biết **còn bao nhiêu chữ cái đang "lệch"** giữa `win` và `need` (Cách 3).

In [10]:
s, p = "cbaebabacd", "abc"
n = len(p)
need = Counter(p)
win = Counter(s[:n])
print(0, s[0:n], win == need)
for i in range(1, len(s) - n + 1):
    win[s[i + n - 1]] += 1          # chữ mới vào bên phải
    win[s[i - 1]] -= 1              # chữ cũ ra bên trái
    print(i, s[i:i + n], "vào", s[i + n - 1], "ra", s[i - 1], "->", win == need)

0 cba True
1 bae vào e ra c -> False
2 aeb vào b ra b -> False
3 eba vào a ra a -> False
4 bab vào b ra e -> False
5 aba vào a ra b -> False
6 bac vào c ra a -> True
7 acd vào d ra b -> False


## 7. Các cách tiếp theo

### Cách 2 — Cửa sổ trượt với hai `Counter`, so sánh mỗi bước

**Giải thích từng đoạn.**

1. Nếu `m < n` thì không thể có cửa sổ nào: trả `[]`.
2. `need = Counter(p)`; `win = Counter(s[:n - 1])` (nạp trước `n - 1` chữ).
3. Với `i` chạy từ `n - 1` đến `m - 1`: thêm `s[i]` (cửa sổ giờ là `[i - n + 1, i]`, đúng `n` chữ); nếu `win == need` thì ghi `i - n + 1`; rồi bớt `s[i - n + 1]` để chuẩn bị bước sau.

In [11]:
def anagrams_two_counters(s, p):
    m, n = len(s), len(p)
    ans = []
    if m < n:
        return ans
    need = Counter(p)
    win = Counter(s[:n - 1])
    for i in range(n - 1, m):
        win[s[i]] += 1
        if win == need:
            ans.append(i - n + 1)
        win[s[i - n + 1]] -= 1
    return ans


for s, p, expected in examples:
    print(anagrams_two_counters(s, p), "(kỳ vọng", expected, ")")

[0, 6] (kỳ vọng [0, 6] )
[0, 1, 2] (kỳ vọng [0, 1, 2] )


In [12]:
# Code nộp LeetCode cho Cách 2
class SolutionTwoCounters:
    def findAnagrams(self, s: str, p: str) -> List[int]:
        m, n = len(s), len(p)
        ans = []
        if m < n:
            return ans
        cnt1 = Counter(p)
        cnt2 = Counter(s[: n - 1])
        for i in range(n - 1, m):
            cnt2[s[i]] += 1
            if cnt1 == cnt2:
                ans.append(i - n + 1)
            cnt2[s[i - n + 1]] -= 1
        return ans


print(SolutionTwoCounters().findAnagrams("abab", "ab"))      # [0, 1, 2]

[0, 1, 2]


**Độ phức tạp Cách 2:** mỗi bước cập nhật `O(1)` và so sánh hai `Counter` có tối đa `26` khoá ⇒ **Time `O(26 · m)`** hay `O(m)` coi `26` là hằng, **Space `O(26)` = `O(1)`** (không kể mảng đáp án).

> Lưu ý phiên bản Python: từ **Python 3.10**, `Counter` so sánh `==` coi khoá thiếu là `0` (nên `Counter(a=0) == Counter()` là `True`). Trước đó hai `Counter` có khoá đếm `0` sẽ **khác** `Counter` không có khoá. LeetCode hiện chạy Python mới nên không sao, nhưng khi tự viết bằng `dict` thường phải rất cẩn thận (xem mục 13).

### Cách 3 — Mảng 26 phần tử + đếm số chữ cái "lệch" (`O(m)`)

**Giải thích từng đoạn.**

1. `diff[k] = (số lần chữ k trong cửa sổ) − (số lần chữ k trong p)` cho `k = 0..25`.
2. `bad` = số chỉ số `k` mà `diff[k] != 0`. Cửa sổ là anagram ⇔ `bad == 0`.
3. Khi thêm chữ `c`: nếu `diff[c]` đang `0` thì sắp lệch (`bad += 1`); tăng `diff[c]`; nếu sau khi tăng thành `0` thì hết lệch (`bad -= 1`). Bớt chữ tương tự.

Nhờ `bad` ta kiểm tra khớp trong `O(1)` thật sự, không cần so sánh cả 26 ô.

In [13]:
def anagrams_diff(s, p):
    m, n = len(s), len(p)
    if m < n:
        return []
    diff = [0] * 26
    for c in p:
        diff[ord(c) - 97] -= 1          # p đóng góp số âm
    bad = sum(1 for x in diff if x != 0)

    def add(ch, delta):
        nonlocal bad
        k = ord(ch) - 97
        if diff[k] == 0:
            bad += 1                    # đang cân bằng, sắp bị lệch
        diff[k] += delta
        if diff[k] == 0:
            bad -= 1                    # vừa cân bằng trở lại

    ans = []
    for i, ch in enumerate(s):
        add(ch, +1)                     # chữ mới vào
        if i >= n:
            add(s[i - n], -1)           # chữ cũ ra
        if i >= n - 1 and bad == 0:
            ans.append(i - n + 1)
    return ans


for s, p, expected in examples:
    print(anagrams_diff(s, p), "(kỳ vọng", expected, ")")

[0, 6] (kỳ vọng [0, 6] )
[0, 1, 2] (kỳ vọng [0, 1, 2] )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionDiff:
    def findAnagrams(self, s: str, p: str) -> List[int]:
        m, n = len(s), len(p)
        if m < n:
            return []
        diff = [0] * 26
        for c in p:
            diff[ord(c) - 97] -= 1
        bad = sum(1 for x in diff if x != 0)
        ans = []
        for i, ch in enumerate(s):
            k = ord(ch) - 97
            if diff[k] == 0:
                bad += 1
            diff[k] += 1
            if diff[k] == 0:
                bad -= 1
            if i >= n:
                k = ord(s[i - n]) - 97
                if diff[k] == 0:
                    bad += 1
                diff[k] -= 1
                if diff[k] == 0:
                    bad -= 1
            if i >= n - 1 and bad == 0:
                ans.append(i - n + 1)
        return ans


print(SolutionDiff().findAnagrams("cbaebabacd", "abc"))      # [0, 6]

[0, 6]


**Độ phức tạp Cách 3:** mỗi ký tự chỉ vào và ra một lần, mỗi thao tác `O(1)` ⇒ **Time `O(m + n)`** (cộng bước dựng `diff` từ `p`), **Space `O(26) = O(1)`**.

### Cách 4 — Cửa sổ co giãn kiểu "còn thiếu" (hai con trỏ `left`, `right`)

**Giải thích từng đoạn.**

1. `need[c]` = số chữ `c` mà `p` yêu cầu; `missing` = tổng số chữ còn **thiếu** (ban đầu `n`).
2. Mở rộng `right`: thêm `s[right]`. Nếu `need[s[right]] > 0` thì chữ này **đang cần** nên `missing -= 1`. Luôn `need[s[right]] -= 1` (có thể âm = dư).
3. Khi cửa sổ vượt độ dài `n` (`right - left + 1 > n`): bỏ `s[left]`: `need[s[left]] += 1`; nếu sau khi cộng `need[s[left]] > 0` thì lại thiếu (`missing += 1`); `left += 1`.
4. Nếu `missing == 0` (và cửa sổ đúng độ dài `n`) thì `left` là một đáp án.

Hai con trỏ ở đây chỉ để cửa sổ **luôn ≤ n**; khi `missing == 0` với đúng `n` chữ thì mọi chữ đều khớp đủ (vì tổng cần là `n`).

In [15]:
def anagrams_missing(s, p):
    need = Counter(p)
    missing = len(p)
    ans = []
    left = 0
    for right, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        if right - left + 1 > len(p):            # cửa sổ quá dài: bỏ chữ trái
            out = s[left]
            need[out] += 1
            if need[out] > 0:
                missing += 1
            left += 1
        if missing == 0:
            ans.append(left)
    return ans


for s, p, expected in examples:
    print(anagrams_missing(s, p), "(kỳ vọng", expected, ")")

[0, 6] (kỳ vọng [0, 6] )
[0, 1, 2] (kỳ vọng [0, 1, 2] )


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionMissing:
    def findAnagrams(self, s: str, p: str) -> List[int]:
        need = Counter(p)
        missing = len(p)
        ans = []
        left = 0
        for right, ch in enumerate(s):
            if need[ch] > 0:
                missing -= 1
            need[ch] -= 1
            if right - left + 1 > len(p):
                out = s[left]
                need[out] += 1
                if need[out] > 0:
                    missing += 1
                left += 1
            if missing == 0:
                ans.append(left)
        return ans


print(SolutionMissing().findAnagrams("abab", "ab"))      # [0, 1, 2]

[0, 1, 2]


**Độ phức tạp Cách 4:** mỗi chỉ số `right` và `left` chỉ tiến, mỗi bước `O(1)` ⇒ **Time `O(m)`**, **Space `O(26)`**.

### Cách 5 — Hash cộng ngẫu nhiên + kiểm chứng

**Giải thích từng đoạn.**

1. Gán mỗi chữ cái một số ngẫu nhiên 60 bit cố định `w[c]` (bằng `random.Random(seed)`).
2. `hp = Σ w[c]` của `p`. Giữ `hs` = tổng `w` của cửa sổ hiện tại, cập nhật `O(1)`: `hs += w[chữ vào] − w[chữ ra]`.
3. Nếu `hs == hp` thì **ứng viên**: kiểm chứng bằng `Counter(window) == Counter(p)` để loại trường hợp trùng hash hiếm gặp, rồi mới nhận.

Khác các cách trước: ta không giữ bảng đếm để so sánh mỗi bước, chỉ giữ **một con số**. Kiểm chứng chỉ chạy khi trùng (thường là khi thật sự khớp), nên kết quả **luôn đúng**; hash chỉ giúp nhanh.

In [17]:
def anagrams_hash(s, p, seed=438):
    m, n = len(s), len(p)
    if m < n:
        return []
    rnd = random.Random(seed)
    w = {c: rnd.getrandbits(60) for c in string.ascii_lowercase}
    hp = sum(w[c] for c in p)
    need = Counter(p)
    hs = sum(w[c] for c in s[:n])
    ans = []
    for i in range(m - n + 1):
        if i > 0:
            hs += w[s[i + n - 1]] - w[s[i - 1]]        # chữ vào, chữ ra
        if hs == hp and Counter(s[i:i + n]) == need:   # kiểm chứng ứng viên
            ans.append(i)
    return ans


for s, p, expected in examples:
    print(anagrams_hash(s, p), "(kỳ vọng", expected, ")")

[0, 6] (kỳ vọng [0, 6] )
[0, 1, 2] (kỳ vọng [0, 1, 2] )


In [18]:
# Code nộp LeetCode cho Cách 5
class SolutionHash:
    def findAnagrams(self, s: str, p: str) -> List[int]:
        m, n = len(s), len(p)
        if m < n:
            return []
        rnd = random.Random(438)
        w = {c: rnd.getrandbits(60) for c in string.ascii_lowercase}
        hp = sum(w[c] for c in p)
        need = Counter(p)
        hs = sum(w[c] for c in s[:n])
        ans = []
        for i in range(m - n + 1):
            if i > 0:
                hs += w[s[i + n - 1]] - w[s[i - 1]]
            if hs == hp and Counter(s[i:i + n]) == need:
                ans.append(i)
        return ans


print(SolutionHash().findAnagrams("cbaebabacd", "abc"))      # [0, 6]

[0, 6]


**Độ phức tạp Cách 5:** mỗi bước `O(1)` cho hash; kiểm chứng `O(n)` chỉ khi trùng hash. Với đầu vào mà **rất nhiều** cửa sổ đều là anagram (ví dụ `s = "aaaa..."`, `p = "aa"`) phần kiểm chứng làm tổng thành **`O(m · n)`** trong trường hợp xấu nhất. **Space `O(26 + n)`** (bảng `w`, `need`, và cửa sổ cắt tạm). Cách này chủ yếu để học ý tưởng hash và vì sao cần kiểm chứng.

## 8. TRACE từng bước

Ta trace Cách 2 (hai bộ đếm) trên `s = "cbaebabacd"`, `p = "abc"`, và Cách 3 (mảng `diff` + `bad`) trên `s = "abab"`, `p = "ab"`.

In [19]:
def trace_two_counters(s, p):
    m, n = len(s), len(p)
    need = Counter(p)
    win = Counter(s[:n - 1])
    ans = []
    show = lambda c: {k: v for k, v in sorted(c.items()) if v}
    print(f"need = {show(need)}, nạp trước n-1 = {n - 1} chữ: win = {show(win)}")
    for i in range(n - 1, m):
        win[s[i]] += 1
        hit = win == need
        if hit:
            ans.append(i - n + 1)
        print(f"i={i}: thêm {s[i]!r} -> cửa sổ [{i - n + 1}..{i}] = {s[i - n + 1:i + 1]!r}, win = {show(win)}, khớp = {hit}")
        win[s[i - n + 1]] -= 1
    return ans


assert trace_two_counters("cbaebabacd", "abc") == [0, 6]

need = {'a': 1, 'b': 1, 'c': 1}, nạp trước n-1 = 2 chữ: win = {'b': 1, 'c': 1}
i=2: thêm 'a' -> cửa sổ [0..2] = 'cba', win = {'a': 1, 'b': 1, 'c': 1}, khớp = True
i=3: thêm 'e' -> cửa sổ [1..3] = 'bae', win = {'a': 1, 'b': 1, 'e': 1}, khớp = False
i=4: thêm 'b' -> cửa sổ [2..4] = 'aeb', win = {'a': 1, 'b': 1, 'e': 1}, khớp = False
i=5: thêm 'a' -> cửa sổ [3..5] = 'eba', win = {'a': 1, 'b': 1, 'e': 1}, khớp = False
i=6: thêm 'b' -> cửa sổ [4..6] = 'bab', win = {'a': 1, 'b': 2}, khớp = False
i=7: thêm 'a' -> cửa sổ [5..7] = 'aba', win = {'a': 2, 'b': 1}, khớp = False
i=8: thêm 'c' -> cửa sổ [6..8] = 'bac', win = {'a': 1, 'b': 1, 'c': 1}, khớp = True
i=9: thêm 'd' -> cửa sổ [7..9] = 'acd', win = {'a': 1, 'c': 1, 'd': 1}, khớp = False


In [20]:
def trace_diff(s, p):
    m, n = len(s), len(p)
    diff = [0] * 26
    for c in p:
        diff[ord(c) - 97] -= 1
    bad = sum(1 for x in diff if x != 0)
    print("diff ban đầu (a..d):", diff[:4], "bad =", bad)
    ans = []
    for i, ch in enumerate(s):
        k = ord(ch) - 97
        bad += diff[k] == 0
        diff[k] += 1
        bad -= diff[k] == 0
        note = f"vào {ch!r}"
        if i >= n:
            k2 = ord(s[i - n]) - 97
            bad += diff[k2] == 0
            diff[k2] -= 1
            bad -= diff[k2] == 0
            note += f", ra {s[i - n]!r}"
        hit = i >= n - 1 and bad == 0
        if hit:
            ans.append(i - n + 1)
        print(f"i={i}: {note}: diff(a..d) = {diff[:4]}, bad = {bad}" + (f"  => đáp án {i - n + 1}" if hit else ""))
    return ans


assert trace_diff("abab", "ab") == [0, 1, 2]

diff ban đầu (a..d): [-1, -1, 0, 0] bad = 2
i=0: vào 'a': diff(a..d) = [0, -1, 0, 0], bad = 1
i=1: vào 'b': diff(a..d) = [0, 0, 0, 0], bad = 0  => đáp án 0
i=2: vào 'a', ra 'a': diff(a..d) = [0, 0, 0, 0], bad = 0  => đáp án 1
i=3: vào 'b', ra 'b': diff(a..d) = [0, 0, 0, 0], bad = 0  => đáp án 2


Bảng trace Cách 2 với `s = "cbaebabacd"`, `p = "abc"` (`n = 3`):

| `i` | Chữ vào | Cửa sổ `[i-2..i]` | Bảng đếm cửa sổ | `== need`? | Đáp án thêm |
|---|---|---|---|---|---|
| 2 | `a` | `cba` | a1 b1 c1 | có | `0` |
| 3 | `e` | `bae` | a1 b1 e1 | không | |
| 4 | `b` | `aeb` | a1 b1 e1 | không | |
| 5 | `a` | `eba` | a1 b1 e1 | không | |
| 6 | `b` | `bab` | a1 b2 | không | |
| 7 | `a` | `aba` | a2 b1 | không | |
| 8 | `c` | `bac` | a1 b1 c1 | có | `6` |
| 9 | `d` | `acd` | a1 c1 d1 | không | |

Chú ý: ngay sau mỗi lần so sánh ta **bớt** chữ ở `i - n + 1` để cửa sổ tiếp theo (sau khi thêm chữ kế tiếp) lại đúng `n` chữ.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (cửa sổ trượt, hai bộ đếm): ngắn, dễ kiểm chứng, `O(m)`.

In [21]:
class Solution:
    def findAnagrams(self, s: str, p: str) -> List[int]:
        m, n = len(s), len(p)
        ans = []
        if m < n:
            return ans
        cnt1 = Counter(p)                    # bảng đếm của p
        cnt2 = Counter(s[: n - 1])           # nạp trước n - 1 chữ đầu
        for i in range(n - 1, m):
            cnt2[s[i]] += 1                  # chữ mới vào: cửa sổ [i-n+1, i]
            if cnt1 == cnt2:
                ans.append(i - n + 1)
            cnt2[s[i - n + 1]] -= 1          # chữ cũ ra khỏi cửa sổ
        return ans


sol = Solution()
for s, p, expected in examples:
    print(repr(s), repr(p), "->", sol.findAnagrams(s, p))

'cbaebabacd' 'abc' -> [0, 6]
'abab' 'ab' -> [0, 1, 2]


## 10. Vì sao thuật toán đúng

- **Đặc trưng anagram:** `u` là anagram của `p` ⇔ `u` và `p` có cùng độ dài và cùng bảng đếm chữ cái. Vì cửa sổ luôn có độ dài đúng `n = len(p)`, ta chỉ cần so sánh bảng đếm.
- **Cập nhật cửa sổ đúng:** cửa sổ `[i, i + n)` và `[i + 1, i + n + 1)` chỉ khác ở chữ `s[i]` (ra) và `s[i + n]` (vào). Cộng/trừ `1` vào đúng hai ô bảng đếm cho ra bảng đếm của cửa sổ mới.
- **Vị trí bắt đầu:** khi vừa thêm `s[i]`, cửa sổ là `[i − n + 1, i]`, bắt đầu tại `i − n + 1`. Vòng lặp `i` từ `n − 1` đến `m − 1` bao đủ mọi `m − n + 1` cửa sổ.
- **Bớt chữ trái sau khi kiểm tra:** để lần thêm tiếp theo cửa sổ lại có đúng `n` chữ. Nếu quên, cửa sổ lớn dần và chẳng bao giờ khớp.
- **Trường hợp `m < n`:** không có cửa sổ độ dài `n` nào; trả `[]`.
- **Cách 3:** `bad` đếm đúng số ô `diff` khác `0`, nên `bad == 0` ⇔ bảng đếm cửa sổ bằng bảng đếm `p`.
- **Cách 4:** `missing` là số chữ `p` còn thiếu; cửa sổ có đúng `n` chữ và `missing == 0` nghĩa là đã phủ đủ `n` chữ cần ⇒ bảng đếm bằng nhau.
- **Cách 5:** luôn kiểm chứng khi trùng hash nên kết quả đúng tuyệt đối; hash chỉ lọc nhanh các cửa sổ chắc chắn không khớp.
- **Đối chiếu độc lập:** mục 12 so với vét cạn "sắp từng cửa sổ" trên hàng nghìn input ngẫu nhiên (cả bảng chữ cái nhỏ để có nhiều khớp).

## 11. Phân tích Time / Space Complexity

Gọi `m = len(s)`, `n = len(p)`, bảng chữ cái `26`.

| Cách | Time | Space |
|---|---|---|
| 1. Sắp từng cửa sổ | `O((m − n + 1) · n log n)` | `O(n)` |
| 2. Hai `Counter` | `O(26 · m)` ≈ `O(m)` | `O(1)` |
| 3. Mảng `diff` + `bad` | `O(m + n)` | `O(1)` |
| 4. Cửa sổ co giãn "missing" | `O(m + n)` | `O(1)` |
| 5. Hash cộng + kiểm chứng | `O(m + n)` kỳ vọng; xấu nhất `O(m · n)` | `O(n)` |

Với `m = n = 3·10^4`, Cách 1 sắp ~1 cửa sổ rất dài; còn khi `n ≈ m/2` chi phí `~ 1.5·10^4 · 1.5·10^4 · log` quá lớn; các Cách 2–4 chỉ cỡ vài chục nghìn thao tác.

## 12. Bộ test

Ta gom các cách vào `impls` (kết quả đều là list **tăng dần** vì ta duyệt từ trái sang phải, nhưng đề cho phép thứ tự bất kỳ nên khi so sánh ta cũng `sorted`).

In [22]:
impls = {
    "two_counters": anagrams_two_counters,
    "diff": anagrams_diff,
    "missing": anagrams_missing,
    "hash": anagrams_hash,
    "leetcode": lambda s, p: Solution().findAnagrams(s, p),
    "submit_two": lambda s, p: SolutionTwoCounters().findAnagrams(s, p),
    "submit_diff": lambda s, p: SolutionDiff().findAnagrams(s, p),
    "submit_missing": lambda s, p: SolutionMissing().findAnagrams(s, p),
    "submit_hash": lambda s, p: SolutionHash().findAnagrams(s, p),
}


def check_all(s, p, expected, label):
    assert anagrams_sort(s, p) == expected, (label, "sort")
    for key, fn in impls.items():
        actual = fn(s, p)
        assert sorted(actual) == expected, (label, key, s, p, actual, expected)
    print("PASS", label)


for idx, (s, p, expected) in enumerate(examples, 1):
    check_all(s, p, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

In [23]:
check_all("a", "a", [0], "một chữ cái giống nhau")
check_all("a", "b", [], "một chữ cái khác nhau")
check_all("a", "ab", [], "p dài hơn s")
check_all("ab", "ab", [0], "s bằng p")
check_all("ba", "ab", [0], "s là hoán vị của p")
check_all("aaaa", "aa", [0, 1, 2], "toàn chữ giống nhau")
check_all("aaaa", "aaa", [0, 1], "p ba chữ giống nhau")
check_all("abcabc", "abc", [0, 1, 2, 3], "các cửa sổ chồng nhau")
check_all("baa", "aa", [1], "chỉ cửa sổ cuối khớp")
check_all("abacbabc", "abc", [1, 2, 3, 5], "nhiều khớp chồng lấn")
check_all("xyz", "abc", [], "không có chữ cái chung")
check_all("aab", "abb", [], "cùng tập chữ nhưng khác số lần")
alpha = string.ascii_lowercase
check_all(alpha + alpha, alpha[::-1], [i for i in range(27)], "p là bảng chữ cái đảo, s lặp hai lần")

PASS một chữ cái giống nhau
PASS một chữ cái khác nhau
PASS p dài hơn s
PASS s bằng p
PASS s là hoán vị của p
PASS toàn chữ giống nhau
PASS p ba chữ giống nhau
PASS các cửa sổ chồng nhau
PASS chỉ cửa sổ cuối khớp
PASS nhiều khớp chồng lấn
PASS không có chữ cái chung
PASS cùng tập chữ nhưng khác số lần
PASS p là bảng chữ cái đảo, s lặp hai lần


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) so sánh **tập hợp** `set` (bỏ qua số lần xuất hiện); (b) **quên bớt chữ bên trái** nên cửa sổ phình ra.

In [24]:
def wrong_set(s, p):
    n = len(p)
    return [i for i in range(len(s) - n + 1) if set(s[i:i + n]) == set(p)]     # SAI


def wrong_no_remove(s, p):
    n = len(p)
    need = Counter(p)
    win = Counter(s[:n - 1])
    ans = []
    for i in range(n - 1, len(s)):
        win[s[i]] += 1
        if win == need:
            ans.append(i - n + 1)
        # SAI: thiếu win[s[i - n + 1]] -= 1
    return ans


print("set:", wrong_set("aab", "abb"), "| đúng:", Solution().findAnagrams("aab", "abb"))
print("không bớt:", wrong_no_remove("abab", "ab"), "| đúng:", Solution().findAnagrams("abab", "ab"))
assert wrong_set("aab", "abb") != []
assert wrong_no_remove("abab", "ab") != [0, 1, 2]

set: [0] | đúng: []
không bớt: [0] | đúng: [0, 1, 2]


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn

Sinh `s`, `p` chữ thường đúng ràng buộc. Ta dùng nhiều bảng chữ cái khác nhau (`"ab"`, `"abc"`, `"abcdef"`, toàn bộ 26 chữ) để có cả trường hợp nhiều khớp lẫn ít khớp, và **cố ý nhúng** vài anagram của `p` vào `s` để chắc chắn có khớp. Thêm thuộc tính: mọi vị trí trả về đều thật sự thoả `sorted(s[i:i+n]) == sorted(p)`, và số đáp án `<= m - n + 1`.

In [25]:
rng = random.Random(438)

for trial in range(2500):
    alpha_k = rng.choice(["ab", "abc", "abcdef", string.ascii_lowercase])
    m = rng.randint(1, 30)
    n = rng.randint(1, 8)
    s = "".join(rng.choice(alpha_k) for _ in range(m))
    p = "".join(rng.choice(alpha_k) for _ in range(n))
    if m >= n and rng.random() < 0.6:                     # nhúng một anagram của p
        pos = rng.randint(0, m - n)
        chars = list(p)
        rng.shuffle(chars)
        s = s[:pos] + "".join(chars) + s[pos + n:]
    expected = anagrams_sort(s, p)
    for key, fn in impls.items():
        assert sorted(fn(s, p)) == expected, (key, s, p)
    assert len(expected) <= max(0, m - n + 1)
    for i in expected:
        assert sorted(s[i:i + n]) == sorted(p)
print("PASS 2500 cặp (s, p) ngẫu nhiên đúng ràng buộc")

for trial in range(30):                                   # kích thước lớn, so các cách nhanh với nhau
    alpha_k = rng.choice(["ab", "abc", string.ascii_lowercase])
    s = "".join(rng.choice(alpha_k) for _ in range(30000))
    p = "".join(rng.choice(alpha_k) for _ in range(rng.randint(1, 6)))
    ref = anagrams_diff(s, p)
    for key in ["two_counters", "missing", "hash", "leetcode"]:
        assert impls[key](s, p) == ref, key
print("PASS 30 input độ dài 3*10^4")

s_big = "a" * 30000
print("s = 'a'*30000, p = 'a'*15000 ->", len(Solution().findAnagrams(s_big, "a" * 15000)), "vị trí")
assert len(Solution().findAnagrams(s_big, "a" * 15000)) == 15001
assert Solution().findAnagrams("a" * 30000, "a" * 30000) == [0]

PASS 2500 cặp (s, p) ngẫu nhiên đúng ràng buộc


PASS 30 input độ dài 3*10^4
s = 'a'*30000, p = 'a'*15000 -> 15001 vị trí


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Không bớt chữ bên trái (cửa sổ phình to)

In [26]:
def sai_khong_bot_chu(s, p):
    n = len(p)
    need = Counter(p)
    win = Counter()
    ans = []
    for i, ch in enumerate(s):
        win[ch] += 1
        if win == need:                 # SAI: win ngày càng lớn, không bao giờ trở lại bằng need
            ans.append(i - n + 1)
    return ans


print("sai:", sai_khong_bot_chu("abab", "ab"), "| đúng:", Solution().findAnagrams("abab", "ab"))

sai: [0] | đúng: [0, 1, 2]


**Vì sao sai:** cửa sổ cần giữ **đúng `n` chữ**. Nếu chỉ thêm mà không bớt, `win` chứa mọi chữ từ đầu đến `i`, chỉ khớp được cửa sổ đầu tiên (nếu có). **Cách sửa:** sau mỗi bước, trừ chữ `s[i - n + 1]` (chữ trái nhất của cửa sổ).

### Lỗi 2: Sai vị trí bắt đầu `i - n + 1`

In [27]:
def sai_vi_tri(s, p):
    n = len(p)
    need = Counter(p)
    win = Counter(s[:n - 1])
    ans = []
    for i in range(n - 1, len(s)):
        win[s[i]] += 1
        if win == need:
            ans.append(i - n)               # SAI: lệch 1, đúng là i - n + 1
        win[s[i - n + 1]] -= 1
    return ans


print("sai:", sai_vi_tri("cbaebabacd", "abc"), "| đúng:", Solution().findAnagrams("cbaebabacd", "abc"))

sai: [-1, 5] | đúng: [0, 6]


**Vì sao sai:** cửa sổ `[l, i]` có độ dài `n` nên `i - l + 1 = n`, tức `l = i - n + 1`. Thiếu `+1` làm mọi đáp án lệch về trái một ô (và có thể ra `-1`). **Cách sửa:** vẽ cửa sổ nhỏ `n = 3` để đếm chỉ số, hoặc kiểm tra `s[l:l+n]` trước khi ghi.

### Lỗi 3: Dùng `set` thay vì bảng đếm

In [28]:
def sai_dung_set(s, p):
    n = len(p)
    return [i for i in range(len(s) - n + 1) if set(s[i:i + n]) == set(p)]


print("sai:", sai_dung_set("aabbb", "aab"), "| đúng:", Solution().findAnagrams("aabbb", "aab"))

sai: [0, 1] | đúng: [0]


**Vì sao sai:** `set` bỏ qua số lần xuất hiện: `{'a','b'}` là tập của cả `"aab"`, `"abb"`, `"bbb"`... **Cách sửa:** dùng `Counter` hoặc mảng đếm (hoặc `sorted`).

### Lỗi 4: Dùng `dict` thường và để lại khoá có đếm `0`

In [29]:
def sai_dict_con_khoa_0(s, p):
    m, n = len(s), len(p)
    need = {}
    for c in p:
        need[c] = need.get(c, 0) + 1
    win = {}
    ans = []
    for i in range(m):
        win[s[i]] = win.get(s[i], 0) + 1
        if i >= n:
            win[s[i - n]] -= 1                 # đếm về 0 nhưng khoá vẫn còn trong dict
        if i >= n - 1 and win == need:         # SAI: {'a': 1, 'x': 0} != {'a': 1}
            ans.append(i - n + 1)
    return ans


print({"a": 1, "x": 0} == {"a": 1}, Counter({"a": 1, "x": 0}) == Counter({"a": 1}))     # False True (Python >= 3.10)
print("sai:", sai_dict_con_khoa_0("xab", "ab"), "| đúng:", Solution().findAnagrams("xab", "ab"))

False True
sai: [] | đúng: [1]


**Vì sao sai:** với `dict` thường, `{'a': 1, 'x': 0}` **khác** `{'a': 1}` dù "về mặt đếm" bằng nhau. Cửa sổ `"ab"` ở vị trí `1` vẫn còn khoá `x` đếm `0` nên không bao giờ khớp. **Cách sửa:** `del win[c]` khi đếm về `0`, hoặc dùng mảng 26 phần tử (so bằng `==`), hoặc `Counter` ở Python ≥ 3.10 (an toàn), hoặc dùng biến `bad`/`missing` như Cách 3, 4.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Tìm mọi chuỗi con là hoán vị / anagram của `p`**" ⇒ cửa sổ trượt **độ dài cố định `len(p)`** + bảng đếm. Cùng họ: *Permutation in String (567)*, *Minimum Window Substring (76)*.
- Khung mẫu: nạp `n − 1` chữ → vòng lặp `i` từ `n − 1`: thêm `s[i]` → kiểm tra → bớt `s[i − n + 1]`.
- Với bảng chữ cái nhỏ dùng **mảng 26** thay `dict`: nhanh hơn và so sánh `==` đơn giản.
- Muốn kiểm tra khớp trong `O(1)` thật sự: giữ biến `bad` (số ô lệch) hoặc `missing` (số chữ còn thiếu).
- Nhớ điều kiện biên `len(s) < len(p)` ⇒ `[]`.
- Khi dùng hash để lọc, **luôn kiểm chứng** hoặc hiểu rõ xác suất đụng độ.

## 15. Biến thể của bài

1. **Permutation in String (567):** chỉ hỏi có ít nhất một anagram của `p` trong `s` không (trả `True/False`).
2. **Minimum Window Substring (76):** cửa sổ ngắn nhất chứa đủ chữ của `t` (cửa sổ **co giãn** thật sự).
3. **Group Anagrams (49):** nhóm các chuỗi theo bảng đếm/chuỗi đã sắp.
4. **Count anagrams chuỗi hoa/thường, Unicode:** dùng `Counter`/`dict` thay mảng 26.
5. **Đếm số cửa sổ** thay vì liệt kê vị trí; hoặc trả về **chính các chuỗi con**.
6. **Anagram với ký tự đại diện `?`** (mỗi `?` thay cho một chữ bất kỳ) — chỉ cần đếm số lệch có thể bù bằng `?`.

In [30]:
def has_permutation(s, p):
    """Biến thể 1 (567): s có chứa hoán vị của p không?"""
    return len(Solution().findAnagrams(s, p)) > 0


def count_anagram_windows(s, p):
    """Biến thể 5: chỉ đếm số cửa sổ."""
    return len(Solution().findAnagrams(s, p))


def group_anagrams(words):
    """Biến thể 3 (49): nhóm theo bảng đếm 26 ô (làm khoá tuple)."""
    groups = {}
    for w_ in words:
        key = [0] * 26
        for c in w_:
            key[ord(c) - 97] += 1
        groups.setdefault(tuple(key), []).append(w_)
    return sorted(sorted(g) for g in groups.values())


print(has_permutation("eidbaooo", "ab"), has_permutation("eidboaoo", "ab"))     # True False
print(count_anagram_windows("abab", "ab"))                                       # 3
print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))
assert has_permutation("eidbaooo", "ab") is True
assert has_permutation("eidboaoo", "ab") is False
assert count_anagram_windows("abab", "ab") == 3
assert group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]) == [["ate", "eat", "tea"], ["bat"], ["nat", "tan"]]

True False
3
[['ate', 'eat', 'tea'], ['bat'], ['nat', 'tan']]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sắp từng cửa sổ | `sorted(s[i:i+n]) == sorted(p)` | `O(m · n log n)` | `O(n)` | oracle kiểm tra, input nhỏ |
| 2. Hai `Counter` | cửa sổ cố định, so bảng đếm | `O(26 · m)` | `O(1)` | **nên nộp**, rõ ràng |
| 3. Mảng `diff` + `bad` | đếm số ô lệch, khớp khi `bad == 0` | `O(m + n)` | `O(1)` | cần `O(1)` thật sự mỗi bước |
| 4. Cửa sổ co giãn "missing" | `missing` = số chữ còn thiếu | `O(m + n)` | `O(1)` | cùng khung với bài Minimum Window |
| 5. Hash cộng + kiểm chứng | tổng trọng số ngẫu nhiên | `O(m + n)` kỳ vọng | `O(n)` | học ý tưởng hash; cần kiểm chứng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"cba"[1:3]`, `sorted("bca")`, `Counter("aab") == Counter("aba")`, `Solution().findAnagrams("abcd", "dc")`, `Solution().findAnagrams("aaa", "aa")`.

**Bài 2 (tính tay).** Với `s = "abacb"`, `p = "abc"`: lập bảng các cửa sổ độ dài `3`, bảng đếm và có khớp hay không; kết quả cuối.

**Bài 3 (điền TODO).** Hoàn thành `anagrams_todo(s, p)` theo Cách 2 (hai bộ đếm 26 phần tử, dùng list thay `Counter`).

In [31]:
def anagrams_todo(s, p):
    m, n = len(s), len(p)
    if m < n:
        return []
    # TODO 1: need = [0] * 26 rồi đếm các chữ của p (ord(c) - 97)
    # TODO 2: win = [0] * 26 rồi đếm n - 1 chữ đầu của s
    ans = []
    # TODO 3: for i in range(n - 1, m): thêm s[i]; nếu win == need thì ans.append(i - n + 1); bớt s[i - n + 1]
    return ans

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `has_permutation2(s, p)` (bài Permutation in String) bằng Cách 4 (`missing`), **dừng sớm** ngay khi `missing == 0` thay vì liệt kê hết.

**Bài 5 (biến thể / mini challenge).** (a) Viết `anagram_starts_with_wildcard(s, p)`: trong `s` có thể có ký tự `'?'` đóng vai **bất kỳ chữ nào**; một cửa sổ được tính nếu có thể thay các `'?'` để thành anagram của `p`. (b) Viết `smallest_window_has_anagram_count(s, p, k)` trả về **chỉ số bắt đầu nhỏ nhất** mà từ đó có ít nhất `k` anagram liên tiếp nhau bắt đầu ở các vị trí `i, i+1, ..., i+k-1` (hoặc `-1`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"ba"`, `['a', 'b', 'c']`, `True`, `[2]` (`"cd"` ở vị trí 2 là anagram của `"dc"`), `[0, 1]`. Bài 3: `need[ord(c) - 97] += 1`.
- **Mức 2 (vừa):** Bài 2: `aba` (a2 b1) không; `bac` (a1 b1 c1) có, vị trí `1`; `acb` có, vị trí `2` ⇒ `[1, 2]`. Bài 4: trong vòng lặp, `if missing == 0: return True`; cuối hàm `return False`.
- **Mức 3 (sâu):** Bài 5(a): đếm các chữ cái thường trong cửa sổ và số `'?'`; cửa sổ hợp lệ nếu **với mọi chữ c**, `win[c] <= need[c]` và số chữ còn thiếu `Σ (need[c] − win[c]) == số '?'`. 5(b): lấy danh sách đáp án `ans` (đã tăng dần), tìm `i` sao cho `i, i+1, ..., i+k-1` đều thuộc `ans` — quét một lần với đếm độ dài dãy liên tiếp.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert "cba"[1:3] == "ba"
assert sorted("bca") == ["a", "b", "c"]
assert Counter("aab") == Counter("aba")
assert Solution().findAnagrams("abcd", "dc") == [2]
assert Solution().findAnagrams("aaa", "aa") == [0, 1]

# Bài 2
assert Solution().findAnagrams("abacb", "abc") == [1, 2]


# Bài 3
def anagrams_todo_ans(s, p):
    m, n = len(s), len(p)
    if m < n:
        return []
    need = [0] * 26
    for c in p:
        need[ord(c) - 97] += 1
    win = [0] * 26
    for c in s[:n - 1]:
        win[ord(c) - 97] += 1
    ans = []
    for i in range(n - 1, m):
        win[ord(s[i]) - 97] += 1
        if win == need:
            ans.append(i - n + 1)
        win[ord(s[i - n + 1]) - 97] -= 1
    return ans


for s, p, expected in examples:
    assert anagrams_todo_ans(s, p) == expected


# Bài 4
def has_permutation2(s, p):
    need = Counter(p)
    missing = len(p)
    left = 0
    for right, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        if right - left + 1 > len(p):
            out = s[left]
            need[out] += 1
            if need[out] > 0:
                missing += 1
            left += 1
        if missing == 0:
            return True
    return False


assert has_permutation2("eidbaooo", "ab") is True
assert has_permutation2("eidboaoo", "ab") is False
assert has_permutation2("a", "ab") is False


# Bài 5
def anagram_starts_with_wildcard(s, p):
    m, n = len(s), len(p)
    need = Counter(p)
    ans = []
    for i in range(m - n + 1):
        window = s[i:i + n]
        q = window.count("?")
        win = Counter(c for c in window if c != "?")
        if all(win[c] <= need[c] for c in win) and sum(need[c] - win[c] for c in need) == q:
            ans.append(i)
    return ans


def smallest_window_has_anagram_count(s, p, k):
    ans = set(Solution().findAnagrams(s, p))
    for i in range(len(s)):
        if all((i + j) in ans for j in range(k)):
            return i
    return -1


assert anagram_starts_with_wildcard("a?cb", "abc") == [0, 1]
assert anagram_starts_with_wildcard("xyz", "abc") == []
assert smallest_window_has_anagram_count("abab", "ab", 3) == 0
assert smallest_window_has_anagram_count("abacb", "abc", 2) == 1
assert smallest_window_has_anagram_count("abab", "ab", 4) == -1
print("Tất cả đáp án bài tập đều đúng.")

Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cửa sổ trượt độ dài cố định với bảng đếm)

```python
class Solution:
    def findAnagrams(self, s: str, p: str) -> List[int]:
        m, n = len(s), len(p)
        ans = []
        if m < n:
            return ans
        cnt1 = Counter(p)
        cnt2 = Counter(s[: n - 1])
        for i in range(n - 1, m):
            cnt2[s[i]] += 1                  # chữ vào
            if cnt1 == cnt2:
                ans.append(i - n + 1)        # vị trí bắt đầu của cửa sổ
            cnt2[s[i - n + 1]] -= 1          # chữ ra
        return ans
```

**Checklist**

- [ ] Cửa sổ luôn có độ dài đúng `len(p)` (thêm một chữ thì bớt một chữ)?
- [ ] Vị trí bắt đầu là `i - n + 1` (không phải `i - n`, không phải `i`)?
- [ ] So sánh **bảng đếm** (không phải `set`)? Đã tránh khoá đếm `0` làm sai khi dùng `dict` thường?
- [ ] Xử lý `len(s) < len(p)` trả `[]`?
- [ ] Đã thử `"cbaebabacd"/"abc"` (`[0,6]`), `"abab"/"ab"` (`[0,1,2]`), `"aaaa"/"aa"` và `"aab"/"abb"` (`[]`)?
- [ ] Skeleton đúng `class Solution` / `def findAnagrams(self, s: str, p: str) -> List[int]`?